In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# load files
os.chdir('..')

raw_ihs_savings_2015 = pd.read_csv('source/householdsavings_15.csv')
raw_ihs_weight_2015 = pd.read_csv('source/householdweight_15.CSV')
raw_ihs_poverty_2015 = pd.read_csv('source/householdpoverty_15.CSV')
raw_finscope_2019 = pd.read_csv('source/finscope_19.csv')
raw_dhs_2020 = pd.read_stata('source/household_19_20.DTA')
raw_findex_2021 = pd.read_csv('source/connectivity_21.csv')
raw_findex_2024 = pd.read_csv('source/connectivity_24.csv')

C:\Users\DELL\AppData\Local\Temp\ipykernel_29208\1196652830.py:7: DtypeWarning: Columns (57,58,173,194,271,329,341,377,381,397,403,480,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,625,633,733,736,741,800,802,806,859,860,861,862,863,864,865,866,867,978,980,1061,1062,1072,1085,1172,1173,1191,1192,1193,1194) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_finscope_2019 = pd.read_csv('source/finscope_19.csv')


In [3]:
raw_ihs_weight_2015

,eanum,hhweight
0,10101,18.111139
1,10103,17.852151
2,10105,17.354952
3,10205,15.203755
4,10208,16.662007
...,...,...
663,86202,10.174545
664,86205,10.078817
665,86210,10.362963
666,86216,28.313095


In [5]:
# 2015 IHS savings account
from pandas import NA


df_savings_2015 = pd.DataFrame()
df_savings_2015['hh_id'] = raw_ihs_savings_2015['hid']
df_savings_2015['settlement'] = raw_ihs_savings_2015['settlement']

df_savings_2015['hh_savings'] = raw_ihs_savings_2015['s7cq1'].map({1.0: 1, 2.0: 0}).fillna(0).astype(int)
df_savings_2015['rural'] = raw_ihs_savings_2015['area'].map({1.0: 0, 2.0: 1}).fillna(0).astype(int)

df_savings_2015['survey_year'] = 2015
df_savings_2015['data_source'] = 'IHS'

df_savings_2015['eanum'] = raw_ihs_savings_2015['eanum']
weight_lookup = raw_ihs_weight_2015[['eanum', 'hhweight']].drop_duplicates(subset=['eanum'])
df_savings_2015 = df_savings_2015.merge(weight_lookup, on='eanum', how='left')
df_savings_2015['weight'] = df_savings_2015['hhweight'].fillna(np.nan)

# clean duplicates
df_savings_2015 = df_savings_2015.drop(columns=['eanum', 'hhweight'])
df_savings_2015['info_count'] = df_savings_2015.notna().sum(axis=1)
df_savings_2015_sorted = df_savings_2015.sort_values(
    by=['hh_id', 'info_count'], 
    ascending=[True, False]
)
df_savings_2015_clean = df_savings_2015_sorted.drop_duplicates(
    subset=['hh_id'], 
    keep='first'
).copy()
df_savings_2015 = df_savings_2015_clean.drop(columns=['info_count'])

In [6]:
# 2015 income
poverty_map = raw_ihs_poverty_2015.drop_duplicates('hid').set_index('hid')['s13q3']
df_savings_2015['hh_income'] = df_savings_2015['hh_id'].astype(int).map(poverty_map)

In [7]:
df_savings_2015

,hh_id,settlement,hh_savings,rural,survey_year,data_source,weight,hh_income
0,1010101,10101,1,0,2015,IHS,18.111139,3
2,1010103,10101,1,0,2015,IHS,18.111139,2
3,1010104,10101,1,0,2015,IHS,18.111139,2
4,1010105,10101,1,0,2015,IHS,18.111139,2
5,1010106,10101,1,0,2015,IHS,18.111139,1
...,...,...,...,...,...,...,...,...
15806,8622216,86231,0,1,2015,IHS,10.126095,3
15807,8622217,86231,0,1,2015,IHS,10.126095,2
15808,8622218,86231,0,1,2015,IHS,10.126095,2
15809,8622219,86231,0,1,2015,IHS,10.126095,1


In [8]:
# 2019 finscope savings
df_savings_2019 = pd.DataFrame()

df_savings_2019['hh_id'] = raw_finscope_2019['ID'] 
df_savings_2019['hh_savings'] = raw_finscope_2019['I4A'].map({"1. Yes": 1, "2. No": 0}).fillna(0).astype(int)
df_savings_2019['rural'] = raw_finscope_2019['HHID9'].map({"Rural": 1, "Urban": 0}).fillna(0).astype(int)
df_savings_2019['settlement'] = raw_finscope_2019['SETTLEMENT'].str.extract(r'^(\d+)')


df_savings_2019['survey_year'] = 2019
df_savings_2019['data_source'] = 'Finscope'
df_savings_2019['weight'] = NA

In [9]:
# 2019 income
finscope_map = raw_finscope_2019.drop_duplicates('ID').set_index('ID')['C9CODE_new']
df_savings_2019['hh_income'] = df_savings_2019['hh_id'].map(finscope_map)

extracted_codes = df_savings_2019['hh_income'].astype(str).str.extract(r'^(\d+)')[0].astype(float)
df_savings_2019['hh_income'] = extracted_codes.replace({99: 0, 15: np.nan})
df_savings_2019['hh_income'] = df_savings_2019['hh_income'].clip(upper=5)

In [10]:
df_savings_2019

,hh_id,hh_savings,rural,settlement,survey_year,data_source,weight,hh_income
0,1,0,0,10201,2019,Finscope,<NA>,1.0
1,26,1,0,10201,2019,Finscope,<NA>,1.0
2,55,0,0,10201,2019,Finscope,<NA>,1.0
3,77,1,0,10201,2019,Finscope,<NA>,1.0
4,102,1,0,10201,2019,Finscope,<NA>,4.0
...,...,...,...,...,...,...,...,...
1465,37066,0,1,86225,2019,Finscope,<NA>,0.0
1466,37099,0,1,86225,2019,Finscope,<NA>,0.0
1467,37121,0,1,86225,2019,Finscope,<NA>,2.0
1468,37140,1,1,86225,2019,Finscope,<NA>,4.0


In [11]:
# 2019-2020 DHS bank account
df_savings_2020 = pd.DataFrame()

df_savings_2020['hh_id'] = raw_dhs_2020['hhid']
df_savings_2020['survey_year'] = raw_dhs_2020['hv007']
df_savings_2020['hh_income'] = raw_dhs_2020['hv270']
df_savings_2020['rural'] = raw_dhs_2020['hv025'].map({2: 1, 1: 0}).fillna(0).astype(int)
df_savings_2020['hh_savings'] = raw_dhs_2020['hv247'].map({1: 1, 0:0}).fillna(0).astype(int)

df_savings_2020['data_source'] = 'DHS'
df_savings_2020['weight'] = raw_dhs_2020['hv005'] / 1_000_0

quintile_labels = {'poorest': 1, 'poorer': 2, 'middle': 3, 'richer': 4, 'richest': 5}
df_savings_2020['hh_income'] = df_savings_2020['hh_income'].map(quintile_labels)

In [12]:
df_savings_2020.head()

,hh_id,survey_year,hh_income,rural,hh_savings,data_source,weight
0,1 1,2019,5,0,0,DHS,17.2113
1,1 6,2019,3,0,0,DHS,17.2113
2,1 10,2019,2,0,0,DHS,17.2113
3,1 14,2019,3,0,0,DHS,17.2113
4,1 18,2019,3,0,0,DHS,17.2113


In [13]:
# 2021 Findex savings
df_savings_2021 = pd.DataFrame()

df_savings_2021['hh_id'] = raw_findex_2021.index.map(lambda x: f"findex_21_{x}")
df_savings_2021['survey_year'] = 2021
df_savings_2021['data_source'] = 'Findex'
df_savings_2021['hh_income'] = raw_findex_2021['inc_q']

df_savings_2021['hh_savings'] = raw_findex_2021['saved'].map({1: 1, 0: 0}).fillna(0).astype(int)
df_savings_2021['rural'] = raw_findex_2021['urbanicity_f2f'].map({1: 1, 2: 0}).fillna(0).astype(int)
df_savings_2021['weight'] = (raw_findex_2021['wgt']*10).fillna(NA)

In [14]:
df_savings_2021

,hh_id,survey_year,data_source,hh_income,hh_savings,rural,weight
0,findex_21_0,2021,Findex,4,1,0,4.965584
1,findex_21_1,2021,Findex,4,0,0,18.787791
2,findex_21_2,2021,Findex,3,0,0,2.202056
3,findex_21_3,2021,Findex,4,0,0,9.393896
4,findex_21_4,2021,Findex,2,0,0,5.049582
...,...,...,...,...,...,...,...
995,findex_21_995,2021,Findex,2,0,0,22.156976
996,findex_21_996,2021,Findex,5,0,1,13.511140
997,findex_21_997,2021,Findex,1,1,1,4.964181
998,findex_21_998,2021,Findex,5,1,0,3.739812


In [15]:
# 2024 Findex savings
df_savings_2024 = pd.DataFrame()

df_savings_2024['hh_id'] = raw_findex_2024.index.map(lambda x: f"findex_24_{x}")
df_savings_2024['survey_year'] = 2024
df_savings_2024['data_source'] = 'Findex'
df_savings_2024['hh_income'] = raw_findex_2024['inc_q']


df_savings_2024['hh_savings'] = raw_findex_2024['saved'].map({1: 1, 0: 0}).fillna(0).astype(int)
df_savings_2024['rural'] = raw_findex_2024['urbanicity'].map({1: 1, 2: 0}).fillna(0).astype(int)
df_savings_2024['weight'] = (raw_findex_2024['wgt']*10).fillna(NA)

In [16]:
df_savings_2024

,hh_id,survey_year,data_source,hh_income,hh_savings,rural,weight
0,findex_24_0,2024,Findex,5,1,0,3.443932
1,findex_24_1,2024,Findex,1,0,1,6.092116
2,findex_24_2,2024,Findex,4,0,0,7.985201
3,findex_24_3,2024,Findex,1,0,0,2.532433
4,findex_24_4,2024,Findex,3,1,0,5.319825
...,...,...,...,...,...,...,...
1003,findex_24_1003,2024,Findex,5,0,0,3.282880
1004,findex_24_1004,2024,Findex,2,1,1,7.774921
1005,findex_24_1005,2024,Findex,1,1,1,6.713281
1006,findex_24_1006,2024,Findex,4,1,0,4.836205


In [17]:
# contruct panel for savings
df_savings = pd.concat([
    df_savings_2015,
    df_savings_2019,
    df_savings_2020,
    df_savings_2021,
    df_savings_2024
], axis=0, ignore_index=True)

df_savings.to_csv('output/did_panel_savings.csv', index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_29208\3614358247.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_savings = pd.concat([


In [18]:
df_savings

,hh_id,settlement,hh_savings,rural,survey_year,data_source,weight,hh_income
0,1010101,10101,1,0,2015,IHS,18.111139,3.0
1,1010103,10101,1,0,2015,IHS,18.111139,2.0
2,1010104,10101,1,0,2015,IHS,18.111139,2.0
3,1010105,10101,1,0,2015,IHS,18.111139,2.0
4,1010106,10101,1,0,2015,IHS,18.111139,1.0
...,...,...,...,...,...,...,...,...
23303,findex_24_1003,NaN,0,0,2024,Findex,3.282880,5.0
23304,findex_24_1004,NaN,1,1,2024,Findex,7.774921,2.0
23305,findex_24_1005,NaN,1,1,2024,Findex,6.713281,1.0
23306,findex_24_1006,NaN,1,0,2024,Findex,4.836205,4.0


In [19]:
# agent distancing 
time_midpoint_map = {
    "1. Less than 10 minutes": 5.0,
    "2. 10 to 20 minutes": 15.0,
    "3. 21 to 30 minutes": 25.5,
    "4. 31 to 60 minutes": 45.5,
    "5. 61 minutes to less than 2 hours": 90.0,
    "6. 2 hours to less than -5 hours": 210.0,
    "98. Don?t Know": np.nan
}

df_distance_2019 = pd.DataFrame()

df_distance_2019['hh_id'] = raw_finscope_2019['ID']
df_distance_2019['survey_year'] = 2019
df_distance_2019['data_source'] = 'FinScope'
df_distance_2019['rural'] = raw_finscope_2019['HHID9'].map({"Rural": 1, "Urban": 0}).fillna(0).astype(int)

raw_finscope_2019['QE1A6_clean'] = raw_finscope_2019['QE1A6'].astype(str).str.strip()
df_distance_2019['agent_time_min'] = raw_finscope_2019['QE1A6_clean'].map(time_midpoint_map)

df_distance_2019['settlement'] = raw_finscope_2019['SETTLEMENT'].str.extract(r'^(\d+)')

display(df_distance_2019.head())

,hh_id,survey_year,data_source,rural,agent_time_min,settlement
0,1,2019,FinScope,0,NaN,10201
1,26,2019,FinScope,0,NaN,10201
2,55,2019,FinScope,0,15.0,10201
3,77,2019,FinScope,0,NaN,10201
4,102,2019,FinScope,0,NaN,10201


In [20]:
df_distance_2019.to_csv('output/did_panel_distance.csv', index=False)